In [ ]:
import os
import sys
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.config import ExperimentConfig
from src.analysis.plots import plot_chsh_results
from src.control.virtual_z import apply_frame
from src.experiments.chsh import run_chsh_experiment

config = ExperimentConfig.load("configs/two_qubit_CZ_zz.yaml")
bell_run = np.load(config.artifact_path("bell_run.npz"))

rho_f = qt.Qobj(bell_run["rho_f"], dims=[[3, 3], [3, 3]])
c1 = float(bell_run["v_phase1"])
c2 = float(bell_run["v_phase2"])
rho_framed = apply_frame(rho_f, c1, c2)
rho_detuned = apply_frame(rho_f, c1 + np.pi / 2, c2)


In [ ]:
# Sanity Check on Ideal |Phi+> State
phi_ideal = (
    qt.tensor(qt.basis(3, 0), qt.basis(3, 0))
    + qt.tensor(qt.basis(3, 1), qt.basis(3, 1))
).unit()
S_ideal, err_ideal, _ = run_chsh_experiment(
    qt.ket2dm(phi_ideal), n_shots=5000, seed=42
)
print(
    f"1. Ideal |Phi+> Sanity Check     -> S = {S_ideal:.3f} ±"
    f" {err_ideal:.3f} (Quantum max: {2*np.sqrt(2):.3f})"
)

# Fridge State WITH Virtual-Z Frame Correction
S_cal, err_cal, E_cal = run_chsh_experiment(rho_framed, n_shots=5000, seed=42)
sigma_violation = (S_cal - 2.0) / err_cal
print(
    f"2. Fridge (Calibrated Virtual-Z) -> S = {S_cal:.3f} ±"
    f" {err_cal:.3f} ({sigma_violation:.1f}σ violation over classical"
    " bound 2.0)"
)

# Fridge State with a deliberately detuned tomography frame (+pi/2 on Q1)
S_detuned, err_detuned, _ = run_chsh_experiment(rho_detuned, n_shots=5000, seed=42)
print(
    f"3. Fridge (Detuned Frame, +pi/2)  -> S = {S_detuned:.3f} ± {err_detuned:.3f}\n"
)

for k, v in E_cal.items():
    ideal_str = "-0.7071" if k == "E(a', b')" else "+0.7071"
    print(f"   {k:10s} = {v:+.4f}  (Ideal: {ideal_str})")


In [ ]:
fig = plot_chsh_results(
    E_cal,
    [S_detuned, S_cal, S_ideal],
    [err_detuned, err_cal, err_ideal],
    comparison_labels=["Detuned\nFrame", "Fridge +\nVirtual-Z", r"Ideal $\Phi^+$"],
)
plt.show()
